# 01 — Fetch Training Data (V2: BTC/USD)
Pull 6 months of 1-minute OHLCV bars from Alpaca Crypto Data API v2 for `BTC/USD`.

**Prerequisites:**
1. Mount your Google Drive.
2. Add `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` to Colab Secrets (🔑 icon in left sidebar).
3. Run all cells top-to-bottom.

**Output:** `/content/drive/MyDrive/algo_trader/data/raw/BTC_USD.parquet`

In [1]:
# Install Alpaca SDK and parquet support
!pip install -q alpaca-py pyarrow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 3.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
RAW_DATA_DIR = '/content/drive/MyDrive/algo_trader/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)
print(f'Output directory: {RAW_DATA_DIR}')

Mounted at /content/drive
Output directory: /content/drive/MyDrive/algo_trader/data/raw


In [3]:
# Load API credentials from Colab Secrets (never hard-code these)
from google.colab import userdata
ALPACA_API_KEY    = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError('Add ALPACA_API_KEY and ALPACA_SECRET_KEY to Colab Secrets first.')
print('Credentials loaded ✓')

TimeoutException: Requesting secret ALPACA_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# V2 universe: single crypto pair
CRYPTO_PAIRS = ['BTC/USD']
print(f'Universe: {CRYPTO_PAIRS}')

In [ ]:
import time
import pandas as pd
from datetime import datetime, timedelta

from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame

client = CryptoHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

END_DATE   = datetime.utcnow().replace(hour=0, minute=0, second=0, microsecond=0)
START_DATE = END_DATE - timedelta(days=182)   # ~6 months

FORWARD_FILL_LIMIT = 5
MAX_RETRIES = 3
RETRY_BACKOFF_BASE = 2

summary = []
failed  = []

for pair in CRYPTO_PAIRS:
    safe_name = pair.replace('/', '_')
    out_path = f'{RAW_DATA_DIR}/{safe_name}.parquet'

    if os.path.exists(out_path):
        print(f'{pair}: already exists — skipping.')
        continue

    df = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request = CryptoBarsRequest(
                symbol_or_symbols=pair,
                timeframe=TimeFrame.Minute,
                start=START_DATE,
                end=END_DATE,
            )
            bars = client.get_crypto_bars(request)
            df = bars.df

            if isinstance(df.index, pd.MultiIndex):
                df = df.loc[pair]
            break

        except Exception as e:
            wait = RETRY_BACKOFF_BASE ** attempt
            print(f'{pair} attempt {attempt} failed ({e}). Retrying in {wait}s...')
            time.sleep(wait)

    if df is None or df.empty:
        print(f'FAILED: {pair}')
        failed.append(pair)
        continue

    df.columns = [c.lower() for c in df.columns]
    df = df[['open', 'high', 'low', 'close', 'volume']]
    df.index = pd.to_datetime(df.index, utc=True)

    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='1min', tz='UTC')
    df = df.reindex(full_idx)
    df = df.ffill(limit=FORWARD_FILL_LIMIT)
    df = df.dropna()

    df.to_parquet(out_path, compression='snappy')

    summary.append({
        'pair': pair,
        'rows': len(df),
        'start': str(df.index.min()),
        'end': str(df.index.max()),
        'path': out_path,
    })

    time.sleep(0.35)

print('\n=== SUMMARY ===')
if summary:
    print(pd.DataFrame(summary).to_string(index=False))
if failed:
    print(f'\n⚠️ Failed pairs ({len(failed)}): {failed}')
else:
    print('\n✅ All pairs fetched successfully.')